In [1]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import numpy as np
import optuna
import gc

# Hyper-parameter tuning 
Our tuning did not cover many hyperparameters due to some issues we faced in the process of getting our pipeline setup. We had some memory leak and value issues that caused us to cut out some other variables; however, we still got good results when training our model on this set of hyperpameters

In [5]:
from utils import UNet, train_model, CorruptionDataset

DATA_DIR = 'data/flickr/resized/'
VAL_DIR = DATA_DIR + 'val/img/'
TRAIN_DIR = DATA_DIR + 'train/img/'
OUT_DIR = 'outputs'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def objective(trial):
    # search space
    lr = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32])

    # image transforms
    transform = transforms.Compose([
        transforms.Resize((256,256)), # maybe drop to 256x256 for speed
        transforms.ToTensor(),
    ])

    # dataset and loader
    dataset = CorruptionDataset(root_dir=VAL_DIR, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    # model
    model = UNet(in_channels=3, out_channels=3).to(DEVICE)

    # training
    val_loss = train_model(model, dataloader, DEVICE, epochs=4, lr=lr)

    # vram cleanup
    model.cpu()
    del model
    del dataloader
    del dataset
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    return val_loss

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=15)

[I 2025-11-14 13:17:25,462] A new study created in memory with name: no-name-23b8930f-6efb-48b9-b204-a16357be70cb


epoch [1/4] batch [0/875] loss: 0.2425
epoch [1/4] batch [50/875] loss: 0.0056
epoch [1/4] batch [100/875] loss: 0.0034
epoch [1/4] batch [150/875] loss: 0.0030
epoch [1/4] batch [200/875] loss: 0.0027
epoch [1/4] batch [250/875] loss: 0.0020
epoch [1/4] batch [300/875] loss: 0.0028
epoch [1/4] batch [350/875] loss: 0.0040
epoch [1/4] batch [400/875] loss: 0.0050
epoch [1/4] batch [450/875] loss: 0.0023
epoch [1/4] batch [500/875] loss: 0.0018
epoch [1/4] batch [550/875] loss: 0.0026
epoch [1/4] batch [600/875] loss: 0.0061
epoch [1/4] batch [650/875] loss: 0.0018
epoch [1/4] batch [700/875] loss: 0.0025
epoch [1/4] batch [750/875] loss: 0.0019
epoch [1/4] batch [800/875] loss: 0.0030
epoch [1/4] batch [850/875] loss: 0.0029
epoch [2/4] batch [0/875] loss: 0.0032
epoch [2/4] batch [50/875] loss: 0.0023
epoch [2/4] batch [100/875] loss: 0.0026
epoch [2/4] batch [150/875] loss: 0.0017
epoch [2/4] batch [200/875] loss: 0.0021
epoch [2/4] batch [250/875] loss: 0.0048
epoch [2/4] batch [300

[I 2025-11-14 14:12:42,137] Trial 0 finished with value: 0.0013318139826878905 and parameters: {'lr': 0.00012474645047095642, 'batch_size': 16}. Best is trial 0 with value: 0.0013318139826878905.


epoch [1/4] batch [0/1750] loss: 0.4101
epoch [1/4] batch [50/1750] loss: 0.0102
epoch [1/4] batch [100/1750] loss: 0.0063
epoch [1/4] batch [150/1750] loss: 0.0042
epoch [1/4] batch [200/1750] loss: 0.0065
epoch [1/4] batch [250/1750] loss: 0.0038
epoch [1/4] batch [300/1750] loss: 0.0046
epoch [1/4] batch [350/1750] loss: 0.0026
epoch [1/4] batch [400/1750] loss: 0.0026
epoch [1/4] batch [450/1750] loss: 0.0045
epoch [1/4] batch [500/1750] loss: 0.0057
epoch [1/4] batch [550/1750] loss: 0.0091
epoch [1/4] batch [600/1750] loss: 0.0022
epoch [1/4] batch [650/1750] loss: 0.0062
epoch [1/4] batch [700/1750] loss: 0.0038
epoch [1/4] batch [750/1750] loss: 0.0021
epoch [1/4] batch [800/1750] loss: 0.0078
epoch [1/4] batch [850/1750] loss: 0.0052
epoch [1/4] batch [900/1750] loss: 0.0033
epoch [1/4] batch [950/1750] loss: 0.0020
epoch [1/4] batch [1000/1750] loss: 0.0027
epoch [1/4] batch [1050/1750] loss: 0.0031
epoch [1/4] batch [1100/1750] loss: 0.0017
epoch [1/4] batch [1150/1750] loss

[I 2025-11-14 15:06:00,311] Trial 1 finished with value: 0.0019491747952997684 and parameters: {'lr': 0.000107635147007434, 'batch_size': 8}. Best is trial 0 with value: 0.0013318139826878905.


...

[I 2025-11-14 21:48:29,395] Trial 12 finished with value: 0.0015951208770275116 and parameters: {'lr': 0.0004277478827066869, 'batch_size': 16}. Best is trial 4 with value: 0.0009956060675904155.


epoch [1/4] batch [0/1750] loss: 0.3199
epoch [1/4] batch [50/1750] loss: 0.0054
epoch [1/4] batch [100/1750] loss: 0.0053
epoch [1/4] batch [150/1750] loss: 0.0058
epoch [1/4] batch [200/1750] loss: 0.0059
epoch [1/4] batch [250/1750] loss: 0.0047
epoch [1/4] batch [300/1750] loss: 0.0023
epoch [1/4] batch [350/1750] loss: 0.0020
epoch [1/4] batch [400/1750] loss: 0.0024
epoch [1/4] batch [450/1750] loss: 0.0036
epoch [1/4] batch [500/1750] loss: 0.0019
epoch [1/4] batch [550/1750] loss: 0.0087
epoch [1/4] batch [600/1750] loss: 0.0027
epoch [1/4] batch [650/1750] loss: 0.0063
epoch [1/4] batch [700/1750] loss: 0.0102
epoch [1/4] batch [750/1750] loss: 0.0021
epoch [1/4] batch [800/1750] loss: 0.0061
epoch [1/4] batch [850/1750] loss: 0.0026
epoch [1/4] batch [900/1750] loss: 0.0028
epoch [1/4] batch [950/1750] loss: 0.0021
epoch [1/4] batch [1000/1750] loss: 0.0052
epoch [1/4] batch [1050/1750] loss: 0.0027
epoch [1/4] batch [1100/1750] loss: 0.0016
epoch [1/4] batch [1150/1750] loss

[I 2025-11-14 22:15:54,532] Trial 13 finished with value: 0.002163234632462263 and parameters: {'lr': 0.0001793860213450315, 'batch_size': 8}. Best is trial 4 with value: 0.0009956060675904155.


epoch [1/4] batch [0/875] loss: 0.2474
epoch [1/4] batch [50/875] loss: 0.0047
epoch [1/4] batch [100/875] loss: 0.0042
epoch [1/4] batch [150/875] loss: 0.0028
epoch [1/4] batch [200/875] loss: 0.0039
epoch [1/4] batch [250/875] loss: 0.0023
epoch [1/4] batch [300/875] loss: 0.0046
epoch [1/4] batch [350/875] loss: 0.0023
epoch [1/4] batch [400/875] loss: 0.0028
epoch [1/4] batch [450/875] loss: 0.0046
epoch [1/4] batch [500/875] loss: 0.0027
epoch [1/4] batch [550/875] loss: 0.0027
epoch [1/4] batch [600/875] loss: 0.0024
epoch [1/4] batch [650/875] loss: 0.0048
epoch [1/4] batch [700/875] loss: 0.0025
epoch [1/4] batch [750/875] loss: 0.0022
epoch [1/4] batch [800/875] loss: 0.0013
epoch [1/4] batch [850/875] loss: 0.0053
epoch [2/4] batch [0/875] loss: 0.0019
epoch [2/4] batch [50/875] loss: 0.0032
epoch [2/4] batch [100/875] loss: 0.0021
epoch [2/4] batch [150/875] loss: 0.0037
epoch [2/4] batch [200/875] loss: 0.0021
epoch [2/4] batch [250/875] loss: 0.0017
epoch [2/4] batch [300

[I 2025-11-14 22:39:51,044] Trial 14 finished with value: 0.0019192200852558017 and parameters: {'lr': 0.00037482458372203414, 'batch_size': 16}. Best is trial 4 with value: 0.0009956060675904155.


# Model Training
Our training is fairly straightfoward. We did have one mistake where we added a model checkpoint saver after starting hyperparameter tuning, so despite the fact that we try to reimport the `train_model()` function, that functionality was not pulled in, resulting in some issues getting saved models

In [5]:
# training
from utils import UNet, CorruptionDataset, train_model

DATA_DIR = 'data/flickr/resized/'
VAL_DIR = DATA_DIR + 'val/img/'
TRAIN_DIR = DATA_DIR + 'train/img/'
OUT_DIR = 'outputs'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 20

# transforms
transform = transforms.Compose([
    transforms.Resize((256,256)), # maybe drop to 256x256 for speed
    transforms.ToTensor(),
])

# get hyperparameters
# best_params = study.best_params
best_params = {'lr': 0.00020749114630415395, 'batch_size': 16} # manually set due to reimport issues

# dataset
dataset = CorruptionDataset(root_dir=TRAIN_DIR, transform=transform)
loader = DataLoader(dataset, batch_size=best_params['batch_size'], shuffle=True)

# model
model = UNet(in_channels=3, out_channels=3)

# train
train_model(model, loader, DEVICE, epochs=EPOCHS, lr=best_params['lr'])

# sample output
x, y = next(iter(loader))
model.eval()
with torch.no_grad():
    preds = model(x.to(DEVICE)).cpu()

save_image(torch.cat([x,preds,y],dim=0), os.path.join(OUT_DIR,'sample_result.png'), nrow=best_params['batch_size'])
print('test complete')

epoch [1/20] batch [0/3500] loss: 0.3690
epoch [1/20] batch [50/3500] loss: 0.0055
epoch [1/20] batch [100/3500] loss: 0.0067
epoch [1/20] batch [150/3500] loss: 0.0054
epoch [1/20] batch [200/3500] loss: 0.0049
epoch [1/20] batch [250/3500] loss: 0.0029
epoch [1/20] batch [300/3500] loss: 0.0043
epoch [1/20] batch [350/3500] loss: 0.0024
epoch [1/20] batch [400/3500] loss: 0.0021
epoch [1/20] batch [450/3500] loss: 0.0019
epoch [1/20] batch [500/3500] loss: 0.0030
epoch [1/20] batch [550/3500] loss: 0.0017
epoch [1/20] batch [600/3500] loss: 0.0040
epoch [1/20] batch [650/3500] loss: 0.0026
epoch [1/20] batch [700/3500] loss: 0.0016
epoch [1/20] batch [750/3500] loss: 0.0031
epoch [1/20] batch [800/3500] loss: 0.0048
epoch [1/20] batch [850/3500] loss: 0.0022
epoch [1/20] batch [900/3500] loss: 0.0029
epoch [1/20] batch [950/3500] loss: 0.0032
epoch [1/20] batch [1000/3500] loss: 0.0019
epoch [1/20] batch [1050/3500] loss: 0.0017
epoch [1/20] batch [1100/3500] loss: 0.0055
epoch [1/20

In [6]:
 # Evaluation 

import math
import torch.nn.functional as F

# Pure PyTorch PSNR
def psnr_torch(img1, img2):
    mse = F.mse_loss(img1, img2)
    if mse == 0:
        return 100
    return 20 * math.log10(1.0 / math.sqrt(mse))

# Pure PyTorch SSIM
def ssim_torch(img1, img2):
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    mu1 = F.avg_pool2d(img1, 3, 1, 0)
    mu2 = F.avg_pool2d(img2, 3, 1, 0)

    mu1_sq = mu1 * mu1
    mu2_sq = mu2 * mu2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.avg_pool2d(img1 * img1, 3, 1, 0) - mu1_sq
    sigma2_sq = F.avg_pool2d(img2 * img2, 3, 1, 0) - mu2_sq
    sigma12 = F.avg_pool2d(img1 * img2, 3, 1, 0) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

    return ssim_map.mean().item()


# Evaluation function
def evaluate_dataset(model, dataset_dir, device="cuda"):
   
    transform = transforms.Compose([
        transforms.Resize((256,256)),
        transforms.ToTensor(),
    ])
   
    dataset = CorruptionDataset(root_dir=dataset_dir, transform=transform)
    loader = DataLoader(dataset, batch_size=1, shuffle=False)

    total_psnr = 0
    total_ssim = 0
    count = 0

    for idx, (corrupted, clean) in enumerate(loader):

        corrupted = corrupted.to(device)
        clean = clean.to(device)

        with torch.no_grad():
            pred = model(corrupted)

        pred = torch.clamp(pred, 0, 1)

        psnr_val = psnr_torch(clean, pred)
        ssim_val = ssim_torch(clean, pred)

        total_psnr += psnr_val
        total_ssim += ssim_val
        count += 1

        if idx % 100 == 0:
            print(f"Processed {idx} images PSNR {psnr_val:.2f} SSIM {ssim_val:.4f}")

    avg_psnr = total_psnr / count
    avg_ssim = total_ssim / count

    print("\nFinal Evaluation Results")
    print("------------------------")
    print(f"Average PSNR {avg_psnr:.3f} dB")
    print(f"Average SSIM {avg_ssim:.4f}")

    return avg_psnr, avg_ssim


# Run evaluation
model.eval()
dataset_dir = "data/flickr/resized/train/img/"
evaluate_dataset(model, dataset_dir)

Processed 0 images PSNR 38.22 SSIM 0.9581
Processed 100 images PSNR 34.41 SSIM 0.9046
Processed 200 images PSNR 33.57 SSIM 0.9063
Processed 300 images PSNR 33.44 SSIM 0.8768
Processed 400 images PSNR 32.19 SSIM 0.9377
Processed 500 images PSNR 40.29 SSIM 0.9829
Processed 600 images PSNR 31.50 SSIM 0.9057
Processed 700 images PSNR 38.89 SSIM 0.9791
Processed 800 images PSNR 34.88 SSIM 0.9148
Processed 900 images PSNR 28.78 SSIM 0.7759
Processed 1000 images PSNR 31.78 SSIM 0.8784
Processed 1100 images PSNR 35.20 SSIM 0.9252
Processed 1200 images PSNR 36.71 SSIM 0.9505
Processed 1300 images PSNR 31.86 SSIM 0.8674
Processed 1400 images PSNR 37.25 SSIM 0.9402
Processed 1500 images PSNR 34.35 SSIM 0.9064
Processed 1600 images PSNR 34.52 SSIM 0.8928
Processed 1700 images PSNR 37.60 SSIM 0.9622
Processed 1800 images PSNR 33.93 SSIM 0.8780
Processed 1900 images PSNR 37.53 SSIM 0.9701
Processed 2000 images PSNR 30.08 SSIM 0.8362
Processed 2100 images PSNR 31.55 SSIM 0.8160
Processed 2200 images 

(34.60192233439547, 0.9051581602107202)